# Zero-Shot Transfer Evaluation on INbreast

**Evaluate optimized models on the target dataset without fine-tuning**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dtobi59/mammography-multiobjective-optimization/blob/main/zero_shot_evaluation.ipynb)

This notebook:
1. Loads Pareto-optimal models from NSGA-III optimization
2. Evaluates them on INbreast (zero-shot, no fine-tuning)
3. Reports image-level and breast-level metrics
4. Compares transfer performance across solutions
5. Saves results to Google Drive

**Prerequisites:**
- Completed NSGA-III optimization (colab_tutorial.ipynb)
- Model checkpoints saved in Google Drive
- Pareto solutions CSV file available

**Author:** David ([@dtobi59](https://github.com/dtobi59))

## 1. Setup Environment

Check GPU and clone repository.

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone the repository
!git clone https://github.com/dtobi59/mammography-multiobjective-optimization.git

# Change to project directory
%cd mammography-multiobjective-optimization

# List files
!ls -la

In [ ]:
# Install required packages
!pip install -q -r requirements.txt

print("\n[SUCCESS] All dependencies installed!")

In [ ]:
# Setup Python path
import sys
import os

project_root = os.getcwd()
print(f"Project root: {project_root}")

if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Added {project_root} to sys.path")

print(f"\nPython sys.path[0]: {sys.path[0]}")
print("[OK] Path setup complete!")

## 2. Mount Google Drive

Access your optimization results and INbreast dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set paths to your data in Google Drive
INBREAST_PATH = "/content/drive/MyDrive/INbreast"
OPTIMIZATION_DIR = "/content/drive/MyDrive/vindr_optimization"
CHECKPOINT_DIR = f"{OPTIMIZATION_DIR}/checkpoints"
RESULTS_DIR = f"{OPTIMIZATION_DIR}/results"

print("\n[SUCCESS] Google Drive mounted!")
print(f"INbreast dataset: {INBREAST_PATH}")
print(f"Optimization results: {OPTIMIZATION_DIR}")

## 3. Load Optimization Results

Load the Pareto front from your NSGA-III optimization.

In [ ]:
import glob
import pandas as pd
from pathlib import Path

# Find Pareto solutions CSV files
pareto_files = sorted(glob.glob(f"{RESULTS_DIR}/pareto_solutions_*.csv"))

print("=" * 80)
print("AVAILABLE OPTIMIZATION RESULTS")
print("=" * 80)

if not pareto_files:
    print("[ERROR] No Pareto solutions found!")
    print(f"\nExpected location: {RESULTS_DIR}")
    print("\nPlease run the optimization notebook first (colab_tutorial.ipynb)")
else:
    print(f"Found {len(pareto_files)} result file(s):\n")
    for i, file in enumerate(pareto_files):
        filename = Path(file).name
        print(f"  {i+1}. {filename}")
    print()
    
    # Load the most recent results
    latest_results = pareto_files[-1]
    print(f"Loading most recent: {Path(latest_results).name}")
    
    pareto_df = pd.read_csv(latest_results)
    print(f"\n[OK] Loaded {len(pareto_df)} Pareto-optimal solutions")
    print("=" * 80)

In [ ]:
# Display Pareto front summary
print("=" * 80)
print("PARETO FRONT SUMMARY")
print("=" * 80)
print(f"Total solutions: {len(pareto_df)}\n")

print("Best solutions for each objective:\n")

best_pr_auc_idx = pareto_df['pr_auc'].idxmax()
print(f"  Best PR-AUC:")
print(f"    Solution ID: {best_pr_auc_idx}")
print(f"    PR-AUC:  {pareto_df.loc[best_pr_auc_idx, 'pr_auc']:.4f}")
print(f"    AUROC:   {pareto_df.loc[best_pr_auc_idx, 'auroc']:.4f}")
print(f"    Brier:   {pareto_df.loc[best_pr_auc_idx, 'brier']:.4f}")
print()

best_auroc_idx = pareto_df['auroc'].idxmax()
print(f"  Best AUROC:")
print(f"    Solution ID: {best_auroc_idx}")
print(f"    PR-AUC:  {pareto_df.loc[best_auroc_idx, 'pr_auc']:.4f}")
print(f"    AUROC:   {pareto_df.loc[best_auroc_idx, 'auroc']:.4f}")
print(f"    Brier:   {pareto_df.loc[best_auroc_idx, 'brier']:.4f}")
print()

best_brier_idx = pareto_df['brier'].idxmin()
print(f"  Best Brier:")
print(f"    Solution ID: {best_brier_idx}")
print(f"    PR-AUC:  {pareto_df.loc[best_brier_idx, 'pr_auc']:.4f}")
print(f"    AUROC:   {pareto_df.loc[best_brier_idx, 'auroc']:.4f}")
print(f"    Brier:   {pareto_df.loc[best_brier_idx, 'brier']:.4f}")
print()

best_robust_idx = pareto_df['robustness_degradation'].idxmin()
print(f"  Best Robustness:")
print(f"    Solution ID: {best_robust_idx}")
print(f"    PR-AUC:  {pareto_df.loc[best_robust_idx, 'pr_auc']:.4f}")
print(f"    AUROC:   {pareto_df.loc[best_robust_idx, 'auroc']:.4f}")
print(f"    Brier:   {pareto_df.loc[best_robust_idx, 'brier']:.4f}")
print(f"    Robustness: {pareto_df.loc[best_robust_idx, 'robustness_degradation']:.4f}")

print("=" * 80)

# Display first few solutions
print("\nFirst 10 solutions:")
pareto_df.head(10)

## 4. Load INbreast Dataset

Load the target dataset for zero-shot evaluation.

In [ ]:
# Update config with INbreast path
with open('config.py', 'r') as f:
    config_content = f.read()

config_content = config_content.replace(
    'INBREAST_PATH = "/content/drive/MyDrive/INbreast"',
    f'INBREAST_PATH = "{INBREAST_PATH}"'
)

with open('config.py', 'w') as f:
    f.write(config_content)

print("[OK] Configuration updated!")
print(f"INbreast path: {INBREAST_PATH}")

In [ ]:
import config
from optimization.nsga3_runner import load_metadata

print("=" * 80)
print("LOADING INBREAST DATASET")
print("=" * 80)

# Load INbreast metadata
inbreast_metadata = load_metadata(
    dataset_name="inbreast",
    dataset_path=config.INBREAST_PATH,
    dataset_config=config.INBREAST_CONFIG
)

print(f"\n[OK] Loaded {len(inbreast_metadata)} images")
print(f"Patients: {inbreast_metadata['patient_id'].nunique()}")
print(f"Breasts: {inbreast_metadata['breast_id'].nunique()}")
print()
print("Label distribution:")
print(inbreast_metadata['label'].value_counts())
print()
print("View distribution:")
print(inbreast_metadata['view'].value_counts())
print("=" * 80)

## 4a. Visualize INbreast Dataset (Optional)

Visualize sample INbreast images to verify data loaded correctly.

**Note:** INbreast images should be in PNG format (converted from DICOM).

This shows:
- Sample malignant and benign cases
- Image metadata (view, laterality, BI-RADS)
- Dataset statistics

In [ ]:
import matplotlib.pyplot as pltimport numpy as npfrom PIL import Imagefrom pathlib import Pathprint("=" * 80)print("INBREAST DATASET VISUALIZATION")print("=" * 80)print()# Select sample imagesmalignant_samples = inbreast_metadata[inbreast_metadata['label'] == 1].head(4)benign_samples = inbreast_metadata[inbreast_metadata['label'] == 0].head(4)import pandas as pdsamples = pd.concat([malignant_samples, benign_samples])print(f"Showing {len(samples)} sample images:")print(f"  Malignant: {len(malignant_samples)}")print(f"  Benign: {len(benign_samples)}")print()# Create figuren_images = len(samples)n_cols = min(4, n_images)n_rows = (n_images + n_cols - 1) // n_colsfig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))if n_rows == 1 and n_cols == 1:    axes = np.array([axes])elif n_rows == 1 or n_cols == 1:    axes = axes.flatten()else:    axes = axes.flatten()fig.suptitle('Sample INbreast Mammography Images', fontsize=16, fontweight='bold')inbreast_image_dir = Path(config.INBREAST_PATH) / config.INBREAST_CONFIG["image_dir"]for idx, (ax, (_, row)) in enumerate(zip(axes, samples.iterrows())):    # Construct image path    if 'image_path' in row and pd.notna(row['image_path']):        img_path = inbreast_image_dir / row['image_path']    else:        img_id = row['image_id']        for ext in ['.png', '.PNG']:            potential_path = inbreast_image_dir / f"{img_id}{ext}"            if potential_path.exists():                img_path = potential_path                break        if img_path.exists():        try:            img = Image.open(img_path)            if img.mode \!= 'L':                img = img.convert('L')                        ax.imshow(img, cmap='gray')                        label_text = 'Malignant' if row['label'] == 1 else 'Benign'            color = 'red' if row['label'] == 1 else 'green'                        title = f"{label_text}"            if 'view' in row and pd.notna(row['view']):                title += f"View: {row['view']}"            if 'laterality' in row and pd.notna(row['laterality']):                title += f" | Lat: {row['laterality']}"            title += ""            if 'birads_original' in row and pd.notna(row['birads_original']):                title += f"BI-RADS: {row['birads_original']}"                        ax.set_title(title, fontsize=10, fontweight='bold', color=color)            ax.axis('off')                    except Exception as e:            ax.text(0.5, 0.5, f'Error loading image', ha='center', va='center', fontsize=8)            ax.axis('off')    else:        ax.text(0.5, 0.5, f'Image not found:{row["image_id"]}', ha='center', va='center', fontsize=10)        ax.axis('off')for idx in range(len(samples), len(axes)):    axes[idx].axis('off')plt.tight_layout()plt.show()print("=" * 80)print("INBREAST DATASET SUMMARY")print("=" * 80)print(f"Total images: {len(inbreast_metadata)}")print(f"Patients: {inbreast_metadata['patient_id'].nunique()}")print(f"Breasts: {inbreast_metadata['breast_id'].nunique()}")print()print("Label distribution:")print(f"  Malignant: {(inbreast_metadata['label'] == 1).sum()} ({(inbreast_metadata['label'] == 1).sum() / len(inbreast_metadata) * 100:.1f}%)")print(f"  Benign: {(inbreast_metadata['label'] == 0).sum()} ({(inbreast_metadata['label'] == 0).sum() / len(inbreast_metadata) * 100:.1f}%)")print()if 'view' in inbreast_metadata.columns:    print("View distribution:")    print(inbreast_metadata['view'].value_counts())print("=" * 80)

## 5. Select Solution to Evaluate

Choose which Pareto-optimal solution to evaluate on INbreast.

In [ ]:
# ============================================================================
# SELECT WHICH SOLUTION TO EVALUATE
# ============================================================================
# Change this to evaluate different solutions:

solution_id = pareto_df['pr_auc'].idxmax()  # Best PR-AUC (default)
# solution_id = pareto_df['auroc'].idxmax()    # Best AUROC
# solution_id = pareto_df['brier'].idxmin()    # Best Brier
# solution_id = 5                               # Specific solution ID

# ============================================================================

selected_solution = pareto_df.iloc[solution_id]

print("=" * 80)
print(f"SELECTED SOLUTION {solution_id}")
print("=" * 80)
print()
print("Hyperparameters:")
print(f"  Learning rate:          {selected_solution['learning_rate']:.6f}")
print(f"  Weight decay:           {selected_solution['weight_decay']:.6f}")
print(f"  Dropout rate:           {selected_solution['dropout_rate']:.4f}")
print(f"  Augmentation strength:  {selected_solution['augmentation_strength']:.4f}")
print(f"  Unfreeze fraction:      {selected_solution['unfreeze_fraction']:.4f}")
print()
print("VinDr (source) performance:")
print(f"  PR-AUC:     {selected_solution['pr_auc']:.4f}")
print(f"  AUROC:      {selected_solution['auroc']:.4f}")
print(f"  Brier:      {selected_solution['brier']:.4f}")
print(f"  Robustness: {selected_solution['robustness_degradation']:.4f}")
print("=" * 80)

## 6. Load Trained Model

Load the model checkpoint from optimization.

In [ ]:
from pathlib import Pathfrom models.resnet import ResNet50WithPartialFineTuningprint("=" * 80)print("LOADING MODEL CHECKPOINT")print("=" * 80)# Find checkpointcheckpoint_path = Path(CHECKPOINT_DIR) / f"eval_{solution_id}" / "best_checkpoint.pt"if not checkpoint_path.exists():    print(f"[ERROR] Checkpoint not found: {checkpoint_path}")    print()    print("Available checkpoint directories:")    for d in sorted(Path(CHECKPOINT_DIR).glob("eval_*")):        print(f"  {d.name}")    raise FileNotFoundError(f"Checkpoint not found for solution {solution_id}")print(f"Loading: {checkpoint_path}")checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)print(f"[OK] Checkpoint loaded")print(f"  Epoch: {checkpoint.get('epoch', 'unknown')}")print(f"  Best PR-AUC: {checkpoint.get('best_pr_auc', 'unknown')}")print()# Create modeldevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")print(f"Device: {device}")model = ResNet50WithPartialFineTuning(    num_classes=1,    dropout_rate=selected_solution['dropout_rate'],    unfreeze_fraction=selected_solution['unfreeze_fraction'],)model.load_state_dict(checkpoint['model_state_dict'])model = model.to(device)model.eval()print("[OK] Model created and weights loaded")print("=" * 80)

## 7. Run Zero-Shot Inference

Evaluate the model on INbreast **without any fine-tuning**.

In [ ]:
from data.dataset import create_dataloaders

# Create INbreast dataloader (no augmentation for evaluation)
inbreast_image_dir = str(Path(config.INBREAST_PATH) / config.INBREAST_CONFIG["image_dir"])

_, inbreast_loader = create_dataloaders(
    train_metadata=inbreast_metadata.head(0),  # Empty train set
    val_metadata=inbreast_metadata,             # All data for evaluation
    image_dir=inbreast_image_dir,
    batch_size=config.BATCH_SIZE,
    augmentation_strength=0.0,  # No augmentation
    num_workers=config.NUM_WORKERS,
)

print(f"[OK] Created dataloader")
print(f"  Total batches: {len(inbreast_loader)}")
print(f"  Batch size: {config.BATCH_SIZE}")
print(f"  Total images: {len(inbreast_metadata)}")

In [ ]:
import numpy as np

print("=" * 80)
print("RUNNING ZERO-SHOT INFERENCE ON INBREAST")
print("=" * 80)
print()

all_preds = []
all_labels = []
all_image_ids = []

with torch.no_grad():
    for batch_idx, (images, labels, metadata) in enumerate(inbreast_loader):
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)
        probs = torch.sigmoid(outputs).squeeze()

        # Store predictions
        all_preds.extend(probs.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())
        all_image_ids.extend(metadata['image_id'])

        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(inbreast_loader):
            print(f"  Processed {batch_idx + 1}/{len(inbreast_loader)} batches", end='\r')

print(f"\n[OK] Inference complete: {len(all_preds)} images")
print("=" * 80)

## 8. Evaluate Performance

Compute metrics at both image-level and breast-level.

In [ ]:
from training.metrics import compute_metrics

print("=" * 80)
print("IMAGE-LEVEL EVALUATION")
print("=" * 80)

all_preds_array = np.array(all_preds)
all_labels_array = np.array(all_labels)

image_metrics = compute_metrics(all_preds_array, all_labels_array)

print(f"\nImages evaluated: {len(all_preds)}")
print()
print("Metrics:")
print(f"  PR-AUC:        {image_metrics['pr_auc']:.4f}")
print(f"  AUROC:         {image_metrics['auroc']:.4f}")
print(f"  Brier Score:   {image_metrics['brier']:.4f}")
print(f"  Brier Null:    {image_metrics['brier_null']:.4f}")
print(f"  Scaled Brier:  {image_metrics['scaled_brier']:.4f}")
print("=" * 80)

In [ ]:
from utils.noisy_or import aggregate_to_breast_level

print("=" * 80)
print("BREAST-LEVEL EVALUATION (NOISY OR AGGREGATION)")
print("=" * 80)

# Create image predictions dictionary
image_predictions = {img_id: pred for img_id, pred in zip(all_image_ids, all_preds)}

# Aggregate to breast level
breast_preds, breast_labels = aggregate_to_breast_level(
    image_predictions=image_predictions,
    metadata=inbreast_metadata
)

print(f"\nAggregated to {len(breast_preds)} breasts")
print()

# Compute breast-level metrics
breast_metrics = compute_metrics(breast_preds, breast_labels)

print("Metrics:")
print(f"  PR-AUC:        {breast_metrics['pr_auc']:.4f}")
print(f"  AUROC:         {breast_metrics['auroc']:.4f}")
print(f"  Brier Score:   {breast_metrics['brier']:.4f}")
print(f"  Brier Null:    {breast_metrics['brier_null']:.4f}")
print(f"  Scaled Brier:  {breast_metrics['scaled_brier']:.4f}")
print("=" * 80)

## 9. Compare Source vs Target Performance

Analyze transfer learning effectiveness.

In [ ]:
print("=" * 80)
print("TRANSFER LEARNING ANALYSIS")
print("=" * 80)
print()
print(f"Model: Solution {solution_id}")
print()
print("Source Dataset (VinDr-Mammo):")
print(f"  PR-AUC: {selected_solution['pr_auc']:.4f}")
print(f"  AUROC:  {selected_solution['auroc']:.4f}")
print(f"  Brier:  {selected_solution['brier']:.4f}")
print()
print("Target Dataset (INbreast) - Image Level:")
print(f"  PR-AUC: {image_metrics['pr_auc']:.4f} ({(image_metrics['pr_auc'] / selected_solution['pr_auc'] - 1) * 100:+.1f}%)")
print(f"  AUROC:  {image_metrics['auroc']:.4f} ({(image_metrics['auroc'] / selected_solution['auroc'] - 1) * 100:+.1f}%)")
print(f"  Brier:  {image_metrics['brier']:.4f} ({(image_metrics['brier'] / selected_solution['brier'] - 1) * 100:+.1f}%)")
print()
print("Target Dataset (INbreast) - Breast Level:")
print(f"  PR-AUC: {breast_metrics['pr_auc']:.4f} ({(breast_metrics['pr_auc'] / selected_solution['pr_auc'] - 1) * 100:+.1f}%)")
print(f"  AUROC:  {breast_metrics['auroc']:.4f} ({(breast_metrics['auroc'] / selected_solution['auroc'] - 1) * 100:+.1f}%)")
print(f"  Brier:  {breast_metrics['brier']:.4f} ({(breast_metrics['brier'] / selected_solution['brier'] - 1) * 100:+.1f}%)")
print()

# Transfer quality assessment
transfer_ratio = breast_metrics['pr_auc'] / selected_solution['pr_auc']
print("Transfer Quality Assessment:")
if transfer_ratio >= 0.9:
    print("  ✓ EXCELLENT - Minimal performance degradation (<10%)")
elif transfer_ratio >= 0.8:
    print("  ✓ GOOD - Acceptable performance retention (>80%)")
elif transfer_ratio >= 0.7:
    print("  ⚠ MODERATE - Noticeable performance drop (70-80%)")
else:
    print("  ⚠ POOR - Significant performance degradation (<70%)")

print("=" * 80)

## 10. Save Results

Save evaluation results to Google Drive.

In [ ]:
from datetime import datetime
import json

print("=" * 80)
print("SAVING EVALUATION RESULTS")
print("=" * 80)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Create evaluation results directory
eval_dir = Path(OPTIMIZATION_DIR) / "inbreast_evaluation"
eval_dir.mkdir(exist_ok=True)

# Prepare results dictionary
eval_results = {
    "timestamp": timestamp,
    "solution_id": int(solution_id),
    "hyperparameters": {
        "learning_rate": float(selected_solution['learning_rate']),
        "weight_decay": float(selected_solution['weight_decay']),
        "dropout_rate": float(selected_solution['dropout_rate']),
        "augmentation_strength": float(selected_solution['augmentation_strength']),
        "unfreeze_fraction": float(selected_solution['unfreeze_fraction']),
    },
    "source_performance": {
        "dataset": "VinDr-Mammo",
        "pr_auc": float(selected_solution['pr_auc']),
        "auroc": float(selected_solution['auroc']),
        "brier": float(selected_solution['brier']),
        "robustness_degradation": float(selected_solution['robustness_degradation']),
    },
    "target_performance": {
        "dataset": "INbreast",
        "image_level": {
            "n_images": len(all_preds),
            "pr_auc": float(image_metrics['pr_auc']),
            "auroc": float(image_metrics['auroc']),
            "brier": float(image_metrics['brier']),
            "brier_null": float(image_metrics['brier_null']),
            "scaled_brier": float(image_metrics['scaled_brier']),
        },
        "breast_level": {
            "n_breasts": len(breast_preds),
            "pr_auc": float(breast_metrics['pr_auc']),
            "auroc": float(breast_metrics['auroc']),
            "brier": float(breast_metrics['brier']),
            "brier_null": float(breast_metrics['brier_null']),
            "scaled_brier": float(breast_metrics['scaled_brier']),
        },
    },
}

# Save JSON
json_path = eval_dir / f"evaluation_solution_{solution_id}_{timestamp}.json"
with open(json_path, 'w') as f:
    json.dump(eval_results, f, indent=2)
print(f"[OK] Saved results: {json_path}")

# Save predictions CSV
predictions_df = pd.DataFrame({
    'image_id': all_image_ids,
    'prediction': all_preds,
    'label': all_labels,
})
csv_path = eval_dir / f"predictions_solution_{solution_id}_{timestamp}.csv"
predictions_df.to_csv(csv_path, index=False)
print(f"[OK] Saved predictions: {csv_path}")

print()
print(f"Results saved to: {eval_dir}")
print("=" * 80)

## Summary

Zero-shot evaluation complete! Key findings are displayed above.

**Next Steps:**
- Evaluate other Pareto solutions (change `solution_id` in Section 5)
- Compare transfer performance across all solutions
- Analyze which hyperparameters lead to better transfer
- Consider ensemble methods combining multiple solutions

**All results saved to Google Drive:**
- JSON file with all metrics
- CSV file with image-level predictions

Access anytime at: `/content/drive/MyDrive/vindr_optimization/inbreast_evaluation/`